### Installation
Install the packages required for executing this notebook.

In [1]:
# Install the packages
! pip3 install --user --no-cache-dir --upgrade "kfp>2" "google-cloud-pipeline-components>2" \
                                        google-cloud-aiplatform

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 234.2 MB/s  0:00:00
   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [google-cloud-aiplatform]  WARNING: The script tb-gcp-uploader is installed in '/home/jupyter/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [google-cloud-pipeline-components]peline-components]


Check the versions of the packages you installed. The KFP SDK version should be >=2

In [2]:
! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform
! python3 -c "import google_cloud_pipeline_components; print('google_cloud_pipeline_components version: {}'.format(google_cloud_pipeline_components.__version__))"

KFP SDK version: 2.17.0
google-cloud-aiplatform==1.165.1
google_cloud_pipeline_components version: 2.22.0


In [3]:
import kfp
import typing
from typing import Dict
from typing import NamedTuple
from kfp import dsl
from kfp.dsl import (Artifact,
                        Dataset,
                        Input,
                        Model,
                        Output,
                        Metrics,
                        ClassificationMetrics,
                        component, 
                        OutputPath, 
                        InputPath)
import google.cloud.aiplatform as aip

#### Pipeline Configurations

In [5]:
#The Google Cloud project that this pipeline runs in.
PROJECT_ID = "your project id"
# The region that this pipeline runs in
REGION = "us-central1"
# Specify a Cloud Storage URI that your pipelines service account can access. The artifacts of your pipeline runs are stored within the pipeline root.
PIPELINE_ROOT = "gs://your temp bucket"

#### Create the Components from Component Specifications 

In [6]:
@dsl.container_component
def data_ingestion(project: str, bucket: str, data_file_name: str,  features: Output[Artifact]): 

    return dsl.ContainerSpec(
        image='us-central1-docker.pkg.dev/your project id/labrepo/dataingestor:0.0.1',
        command=[
            'python3', '/pipelines/component/src/component.py'
        ],
        args=['--project_id',project,'--bucket',bucket,'--file_name',data_file_name,'--feature_path', features.path])

In [7]:
@dsl.container_component
def mlp_training(project: str, features: Input[Artifact], model_bucket: str,  metrics: OutputPath(str)):
    
    return dsl.ContainerSpec(
        image='us-central1-docker.pkg.dev/your project id/labrepo/mlptrainer:0.0.1',
        command=[
            'python3', '/pipelines/component/src/component.py'
        ],
        args=['--project_id',project,'--feature_path',features.path,'--model_repo',model_bucket,'--metrics_path', metrics])

#### Define the Pipeline

In [8]:
# Define the workflow of the pipeline.
@kfp.dsl.pipeline(
    name="diabetes-predictor-mlp")
def pipeline(project_id: str, data_bucket: str, trainset_filename: str, model_repo: str):
    
    # The first step    
    di_op = data_ingestion(
        project=project_id,
        bucket=data_bucket,
        data_file_name=trainset_filename
    )

    # The second step 
    training_op = mlp_training(
        project=project_id,
        model_bucket=model_repo,       
        features=di_op.outputs['features']
    )

#### Compile the Pipeline

In [9]:
from kfp import compiler
compiler.Compiler().compile(pipeline_func=pipeline,
        package_path='diabetes_predictor_mlp.yaml')

#### Run the Pipeline

In [10]:
import google.cloud.aiplatform as aip

# Before initializing, make sure to set the GOOGLE_APPLICATION_CREDENTIALS
# environment variable to the path of your service account.
aip.init(
    project=PROJECT_ID,
    location=REGION,
)


job = aip.PipelineJob(
    display_name="diabetes-predictor-mlp-pipeline",
    template_path="diabetes_predictor_mlp.yaml",
    enable_caching=False,
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        'project_id': PROJECT_ID,
        'data_bucket': 'de2026_data', # make sure to use your data bucket name 
        'trainset_filename': 'training_set.csv',
        'model_repo':'de2026_models' # make sure to use your model bucket name 
    }
)

job.run()

Error when trying to get or create a GCS bucket for the pipeline output artifacts
Traceback (most recent call last):
  File "/opt/micromamba/lib/python3.12/site-packages/google/cloud/aiplatform/pipeline_jobs.py", line 464, in submit
    gcs_utils.create_gcs_bucket_for_pipeline_artifacts_if_it_does_not_exist(
  File "/opt/micromamba/lib/python3.12/site-packages/google/cloud/aiplatform/utils/gcs_utils.py", line 381, in create_gcs_bucket_for_pipeline_artifacts_if_it_does_not_exist
    raise ValueError(
ValueError: Output artifacts bucket "de2026_temp" exists but does not belong to project "de2026-508411". This may indicate a bucket squatting attack. Please provide an explicit output_artifacts_gcs_dir parameter or configure one via aiplatform.init(output_artifacts_gcs_dir='gs://your-bucket').
Creating PipelineJob
PipelineJob created. Resource name: projects/529629719494/locations/us-central1/pipelineJobs/diabetes-predictor-mlp-20261010105052
To use this PipelineJob in another session:
pipe